# Task1 流程复盘：用 TuShare 构建寒武纪 A 股数据展示项目

这个 notebook 记录本项目从需求拆解到数据获取、CSV 保存、静态网页生成、校验与 GitHub 发布的完整可学习流程。

说明：这里记录的是可公开分享的工程思路、关键决策和复现步骤，不包含任何真实 token，也不包含不可公开的内部推理。

## 1. 目标

用户希望完成一条完整的数据项目流水线：

1. 查询中国 A 股“寒武纪”过去一年的交易日数据。
2. 保存未复权数据和复权数据两份 CSV。
3. 绘制每日收盘价曲线。
4. 生成静态 HTML 文件用于展示。
5. 将项目成果保存到根目录 `Task1/` 文件夹。
6. 创建与根目录同名的 GitHub public 仓库并发布项目。

最终项目路径：`Task1/`

最终 GitHub 仓库：`https://github.com/LizPink/PKU-WorkShop-202607`

## 2. 输入与安全边界

### 数据源

用户提供了 TuShare token 和 TuShare MCP Server URL。两者的关系可以理解为：

- token 是访问 TuShare 数据的凭据。
- MCP Server 是一种封装后的访问方式，URL 中也带有 token。
- 如果只是写可复现脚本，直接调用 TuShare HTTP API 更简单，也更容易放进项目脚本。

### 凭据处理

真实 token 只用于本地拉取数据，没有写进：

- Python 脚本
- CSV 数据
- HTML 页面
- README
- GitHub 仓库

项目只保留 `.env.example`，用来提示读者通过环境变量或本地 `.env` 提供 token。

## 3. 项目结构设计

用户补充要求所有成果放在 `Task1/` 文件夹下，所以最终结构是：

```text
Task1/
  data/        # 三份 CSV 数据
  scripts/     # 可复现数据脚本
  web/         # 静态 HTML 页面
  README.md    # 项目说明
  requirements.txt
  .env.example
  .gitignore
```

这样的结构有两个好处：

- 数据、脚本、展示页面清楚分层。
- 根目录中原有 Word 文档不会被误提交到 public 仓库。

## 4. 股票、时间区间与复权口径

### 股票识别

寒武纪的 TuShare 股票代码使用：`688256.SH`。

### 时间区间

用户说“过去一年”，执行时按自然日区间处理：

- 请求开始日：`2025-07-04`
- 请求结束日：`2026-07-04`

TuShare 返回的最新交易日是 `2026-07-03`，因为 `2026-07-04` 是周六，不是交易日。

### 复权方式

本项目保留两套数据：

- 未复权：TuShare `daily` 接口原始行情。
- 前复权：使用 TuShare `adj_factor` 复权因子计算。

前复权是 A 股行情软件中最常见的展示口径之一。公式为：

```text
前复权价格 = 未复权价格 * 当日复权因子 / 区间最新复权因子
```

这样做的特点是：最新价格等于真实交易价格，历史价格被调整，走势更连续。

## 5. 数据拉取脚本的核心逻辑

脚本位置：`Task1/scripts/fetch_tushare_cambricon.py`

核心步骤：

1. 从环境变量 `TUSHARE_TOKEN` 或本地 `.env` 读取 token。
2. 调用 TuShare `stock_basic` 校验股票元数据。
3. 调用 TuShare `daily` 获取未复权日线数据。
4. 调用 TuShare `adj_factor` 获取复权因子。
5. 合并日线与复权因子。
6. 计算前复权开高低收。
7. 输出三份 CSV。
8. 生成静态 HTML 页面。

复现命令示例：

```powershell
$env:TUSHARE_TOKEN = "your_token_here"
python .\scripts\fetch_tushare_cambricon.py --start-date 20250704 --end-date 20260704
```

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != "Task1":
    ROOT = ROOT / "Task1"

DATA_DIR = ROOT / "data"
WEB_DIR = ROOT / "web"

files = {
    "unadjusted": DATA_DIR / "cambricon_688256_SH_daily_unadjusted_20250704_20260704.csv",
    "qfq": DATA_DIR / "cambricon_688256_SH_daily_qfq_20250704_20260704.csv",
    "combined": DATA_DIR / "cambricon_688256_SH_daily_combined_20250704_20260704.csv",
    "html": WEB_DIR / "index.html",
}

for label, path in files.items():
    print(f"{label:10s}", path, "exists=", path.exists())

## 6. 检查 CSV 输出

这里读取已经生成的 CSV，确认三份数据都存在，且交易日数量一致。

In [ ]:
unadjusted = pd.read_csv(files["unadjusted"])
qfq = pd.read_csv(files["qfq"])
combined = pd.read_csv(files["combined"])

summary_rows = pd.DataFrame([
    {"dataset": "unadjusted", "rows": len(unadjusted), "first_date": unadjusted["date"].iloc[0], "last_date": unadjusted["date"].iloc[-1]},
    {"dataset": "qfq", "rows": len(qfq), "first_date": qfq["date"].iloc[0], "last_date": qfq["date"].iloc[-1]},
    {"dataset": "combined", "rows": len(combined), "first_date": combined["date"].iloc[0], "last_date": combined["date"].iloc[-1]},
])

summary_rows

In [ ]:
assert len(unadjusted) == len(qfq) == len(combined) == 242
assert combined["date"].iloc[0] == "2025-07-04"
assert combined["date"].iloc[-1] == "2026-07-03"
assert {"close", "qfq_close", "adj_factor"}.issubset(combined.columns)

print("CSV checks passed.")

## 7. 检查关键指标

网页顶部的指标来自合并数据。这里复算一次，方便学习数据展示页的指标来源。

In [ ]:
first = combined.iloc[0]
latest = combined.iloc[-1]

metrics = {
    "first_trade_date": first["date"],
    "latest_trade_date": latest["date"],
    "latest_close": latest["close"],
    "latest_qfq_close": latest["qfq_close"],
    "latest_pct_chg": latest["pct_chg"],
    "raw_return_pct": (latest["close"] / first["close"] - 1) * 100,
    "qfq_return_pct": (latest["qfq_close"] / first["qfq_close"] - 1) * 100,
}

pd.Series(metrics)

## 8. 静态 HTML 展示页如何设计

页面位置：`Task1/web/index.html`

设计目标：

- 打开即用，不依赖后端服务。
- 数据内嵌在 HTML 中，下载链接指向项目 CSV。
- 页面包含四个关键指标卡片。
- SVG 折线图展示未复权与前复权每日收盘价。
- 分段按钮支持切换“双线 / 未复权 / 前复权”。
- 表格按时间倒序展示交易日明细。
- 方法说明区写清楚数据来源和前复权公式。

因为用户要求静态 HTML，所以没有使用 React/Vite 这类前端工程；一个自包含 HTML 更适合这个任务。

In [ ]:
html = files["html"].read_text(encoding="utf-8")

checks = {
    "contains_title": "寒武纪每日收盘价" in html,
    "contains_svg_chart": "priceChart" in html,
    "contains_download_links": "未复权 CSV" in html and "前复权 CSV" in html,
    "contains_mcp_token_url": "api.tushare.pro/mcp/?token=" in html,
}

checks

In [ ]:
assert "寒武纪每日收盘价" in html
assert "priceChart" in html
assert "未复权 CSV" in html and "前复权 CSV" in html
assert "api.tushare.pro/mcp/?token=" not in html

print("HTML checks passed and no tokenized MCP URL found.")

## 9. 浏览器 QA 做了什么

为了确认页面不是“文件生成了但打不开”，使用本地临时服务打开页面：

```powershell
python -m http.server 8765 --bind 127.0.0.1
```

然后在浏览器里检查：

- 桌面宽度下图表非空。
- 手机宽度下没有横向溢出。
- 关键指标文字不溢出卡片。
- 下载按钮文字不溢出。
- 页面没有 JavaScript error。

QA 截图只保存在本地 `Task1/web/qa/`，并被 `.gitignore` 忽略，没有发布到 GitHub。

## 10. GitHub 发布流程

发布目标：创建 public 仓库，仓库名与根目录同名：`PKU-WorkShop-202607`。

执行策略：

1. 初始化本地 Git 仓库。
2. 只暂存 `Task1/`，不暂存根目录 Word 文档。
3. 本地提交：`Add Cambricon TuShare dashboard`。
4. 使用 GitHub CLI 创建 public 仓库。
5. 尝试 `git push`。
6. 本机 Git HTTPS 推送被网络层阻断后，改用 GitHub Contents API 发布同一批文件。
7. 远端读回目录，确认 CSV、脚本和 HTML 都已存在。

远端仓库：`https://github.com/LizPink/PKU-WorkShop-202607`

## 11. 复现清单

如果你之后想从零复现，可以按这个顺序做：

1. 创建项目目录：`Task1/`。
2. 创建子目录：`data/`、`scripts/`、`web/`。
3. 写 `.gitignore`，忽略 `.env`、缓存、QA 截图等。
4. 写数据脚本，让 token 从环境变量或 `.env` 读取。
5. 调用 TuShare `daily` 和 `adj_factor`。
6. 计算前复权价格。
7. 输出未复权、前复权、合并三份 CSV。
8. 生成静态 HTML 页面。
9. 检查 CSV 行数、日期范围、HTML 是否包含图表和下载链接。
10. 用本地服务打开页面，做桌面和移动端 QA。
11. 只提交项目成果，不提交凭据和无关文件。
12. 创建 GitHub public 仓库并发布。

## 12. 常见风险点

- **token 泄露**：不要把 token 写进脚本、README、HTML、notebook 或 GitHub。真实 token 如果在聊天或日志中出现过，建议后续轮换。
- **日期误解**：自然日结束日不一定是交易日。本项目请求到 `2026-07-04`，但最新交易日是 `2026-07-03`。
- **复权口径混淆**：未复权价格是真实历史交易价格；前复权价格更适合画连续走势，但历史价格会被调整。
- **Git 发布范围过大**：public 仓库中不要误提交根目录已有文档或临时文件。
- **静态页面路径**：HTML 下载链接要和 CSV 相对路径匹配，否则本地打开或 GitHub 浏览时会断链。

## 13. 本 notebook 的验证状态

当前 notebook 使用标准 `.ipynb` JSON 结构生成。

在本环境中，`nbformat`、`nbclient` 和 `matplotlib` 不可用，所以没有用 `nbclient` 做完整 notebook 执行写回。为保持可运行性，代码单元只依赖 `pandas` 和标准库；上面的检查逻辑可以在有 Jupyter 环境时直接运行。

建议本地完整执行命令：

```powershell
python -m jupyter nbconvert --execute --to notebook --inplace .\Task1_process_walkthrough.ipynb
```

如果缺少 Jupyter 相关依赖，可先安装：

```powershell
pip install nbformat nbclient jupyter
```